# **Preliminary EDA: Raw Data Overview**

This notebook provides a preliminary exploratory data analysis (EDA) of the raw, unprocessed data. The goal is to understand two key aspects of the source material:

1.  **Text Length Distribution:** To see the raw character length of the corporate reports, justifying the need for a long-context model.
2.  **Original Score Distribution:** To analyze the nature of the expert scores for all 9 ESG criteria, which provides the rationale for our hybrid model approach.

### **1. Setup and Data Loading**

In [1]:
import pandas as pd
import plotly.express as px
import json

# --- Configuration ---
RAW_SCORES_PATH = '../data/expert_scores.csv'
CLEAN_JSONL_PATH = '../data/processed/reports_texts_clean.jsonl'

In [2]:
# --- Load and Merge Data ---
# Load the expert scores
try:
    scores_df = pd.read_csv(RAW_SCORES_PATH, encoding='utf-8', sep=";")
    scores_df.columns = scores_df.columns.str.strip()
    scores_df.set_index('Raport', inplace=True)
    # Convert comma decimals to dots for all score columns
    for col in scores_df.columns:
        if scores_df[col].dtype == 'object':
            scores_df[col] = scores_df[col].str.replace(',', '.').astype(float)
    print(f"✅ Loaded {len(scores_df)} records from expert scores CSV.")
except FileNotFoundError:
    print(f"❌ Expert scores file not found at: {RAW_SCORES_PATH}")
    scores_df = pd.DataFrame()

# Load text lengths
reports_list = []
try:
    with open(CLEAN_JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            reports_list.append(json.loads(line))
    text_lengths_df = pd.DataFrame({
        'Raport': report.get('nazwa_raportu', '').strip(),
        'text_length': len(report.get('text', ''))
    } for report in reports_list).set_index('Raport')
    print(f"✅ Loaded {len(text_lengths_df)} records from cleaned JSONL.")
    
    # Merge the two dataframes
    if not scores_df.empty:
        raw_data_df = scores_df.join(text_lengths_df, how='inner')
        print(f"✅ Merged data into a single DataFrame with {len(raw_data_df)} final records.")
except FileNotFoundError:
    print(f"❌ Cleaned JSONL file not found at: {CLEAN_JSONL_PATH}")
    raw_data_df = pd.DataFrame()

✅ Loaded 393 records from expert scores CSV.
✅ Loaded 393 records from cleaned JSONL.
✅ Merged data into a single DataFrame with 393 final records.


### **2. Analysis of Raw Text Lengths**

This histogram shows the distribution of character counts in the corporate reports after initial cleaning.

In [3]:
if not raw_data_df.empty:
    fig = px.histogram(
        raw_data_df,
        x='text_length',
        title='<b>Distribution of Raw Report Length (in Characters)</b>',
        labels={'text_length': 'Report Length (Number of Characters)'},
        nbins=50
    )
    fig.update_layout(
        yaxis_title='Number of Reports',
        title_x=0.5,
        bargap=0.1
    )
    fig.show()

#### **Insight:**
The distribution is right-skewed, with a majority of reports having a length between 0 and 400,000 characters. The long tail confirms the presence of very large reports, underscoring the necessity of a model architecture capable of handling long documents, such as Longformer.

### **3. Analysis of Original Expert Scores**

This section analyzes the structure of the original expert scores. It reveals a crucial distinction between two types of criteria in our dataset, which directly justifies our hybrid model approach.

In [4]:
if not raw_data_df.empty:
    melted_df = raw_data_df.melt(
        id_vars=['text_length'], 
        value_vars=[str(i) for i in range(1, 10)], 
        var_name='Criterion', 
        value_name='Score'
    )
    
    # --- PLOT 1: BINARY CRITERIA ---
    binary_criteria_df = melted_df[melted_df['Criterion'] != '3']
    
    print("--- Distribution for Binary Criteria (Score 0 or 1) ---")
    fig1 = px.histogram(
        binary_criteria_df,
        x='Score',
        facet_col='Criterion',
        facet_col_wrap=4,
        height=500,
        title='<b>Distribution of Original Scores for Binary Criteria</b>',
        labels={'count': 'Number of Reports', 'Score': 'Expert Score (0 or 1)'}
    )
    fig1.update_layout(title_x=0.5, yaxis_title='')
    fig1.for_each_annotation(lambda a: a.update(text=f"<b>Criterion {a.text.split('=')[-1]}</b>"))
    fig1.update_xaxes(type='category')
    fig1.show()

    # --- PLOT 2: MULTI-LEVEL CRITERION ---
    multi_level_criterion_df = melted_df[melted_df['Criterion'] == '3']
    
    print("\n--- Distribution for Multi-Level Criterion (Criterion 3) ---")
    fig2 = px.histogram(
        multi_level_criterion_df,
        x='Score',
        title='<b>Distribution of Original Scores for Multi-Level Criterion 3</b>',
        labels={'count': 'Number of Reports', 'Score': 'Expert Score (0 to 1)'},
        text_auto=True
    )
    fig2.update_layout(title_x=0.5, bargap=0.2)
    fig2.update_xaxes(type='category')
    fig2.show()

--- Distribution for Binary Criteria (Score 0 or 1) ---



--- Distribution for Multi-Level Criterion (Criterion 3) ---


#### **Key Insights & Justification for Hybrid Model:**

The visualizations clearly reveal two distinct types of scoring methodologies in our raw data:

1.  **Binary Criteria (0 or 1):** As shown in the first grid of plots, eight of the criteria (1, 2, 4, 5, 6, 7, 8, 9) are fundamentally binary. The expert scores are exclusively 0 or 1, representing a clear "no" or "yes" assessment. These tasks are well-suited for a standard binary/multi-label classification model.

2.  **Multi-Level, Hierarchical Criterion (Criterion 3):** The second plot highlights that Criterion 3 stands alone. It has a granular, multi-level scoring system (0, 0.25, 0.5, 0.75, 1.0), reflecting a hierarchical assessment of disclosure maturity. This type of structured, rule-based evaluation is not an ideal fit for a probabilistic deep learning model.

**This fundamental distinction is the primary justification for our hybrid system architecture:**
-   The **Machine Learning Model** (`Longformer`) is tasked with the six most nuanced binary classification problems.
-   A deterministic **Rule-Based System** is used to handle the three criteria (including the multi-level Criterion 3) that are based on precise, verifiable conditions, ensuring 100% accuracy and transparency for those specific tasks.